# CNN Autoencoder — RGB Colour Image Super-Resolution

**Course**: CPSC 5440 — Spring 2026  
**Task**: Upscale cat/dog images from **28×28** to **56×56** in full colour using a convolutional autoencoder.  
**Dataset**: Kaggle Dogs vs. Cats (25,000 train / 12,500 test)

### Difference from the grayscale notebook

| Aspect | Grayscale | RGB (this notebook) |
|--------|-----------|---------------------|
| PIL mode | `convert('L')` | `convert('RGB')` |
| Tensor shape | `(N, 1, H, W)` via `reshape` | `(N, 3, H, W)` via `transpose(0,3,1,2)` |
| First conv | `Conv2d(1, 32, ...)` | `Conv2d(3, 32, ...)` |
| Last conv | `Conv2d(16, 1, ...)` | `Conv2d(16, 3, ...)` |
| Display | `cmap='gray'` | `.transpose(1,2,0).clip(0,1)` |

> The decoder architecture (bottleneck and upsampling path) is **identical** to the grayscale version.

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision pillow numpy matplotlib torchinfo gdown --quiet

## 2. Download Dataset

Downloads the dataset archive from Google Drive using `gdown`, then extracts it with `7z`.

In [ ]:
import os

URL    = "https://drive.google.com/file/d/1NL36xOWsziZj8grWtm67Yn7EewD6VP0d/view?usp=sharing"
OUTPUT = "data.7z"

!gdown --fuzzy "{URL}" -O "{OUTPUT}"
print("✅ Dataset downloaded")

EXTRACT_PATH = "/content/data"
os.makedirs(EXTRACT_PATH, exist_ok=True)

!7z x "{OUTPUT}" -o"{EXTRACT_PATH}" -y > /dev/null

DATA_ROOT = EXTRACT_PATH
print("✅ Dataset ready")
print(f'Train images: {len(os.listdir(os.path.join(DATA_ROOT, "train")))}')
print(f'Test images : {len(os.listdir(os.path.join(DATA_ROOT, "test1")))}')

## 3. Imports

In [ ]:
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import glob
import numpy as np
from torchinfo import summary
from PIL import Image
import matplotlib.pyplot as plt

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

## 4. Load and Cache Training Data

RGB images are loaded with `convert('RGB')`, giving shape `(N, H, W, 3)` (HWC order).  
`numpy.transpose(0, 3, 1, 2)` converts to `(N, 3, H, W)` (CHW order) as required by PyTorch `Conv2d`.  
Cached with `_rgb` suffix so grayscale and RGB caches never overwrite each other.

In [ ]:
PROCESSED    = 'data/processed'
os.makedirs(PROCESSED, exist_ok=True)

cache_xtrain = os.path.join(PROCESSED, 'x_train_rgb.npy')
cache_ytrain = os.path.join(PROCESSED, 'y_train_rgb.npy')

if os.path.exists(cache_xtrain) and os.path.exists(cache_ytrain):
    x_train = np.load(cache_xtrain)
    y_train = np.load(cache_ytrain)
    print(f'Loaded from cache  x_train={x_train.shape}  y_train={y_train.shape}')
else:
    x_list, y_list = [], []
    files = glob.glob(os.path.join(DATA_ROOT, 'train', '*.jpg'))
    for i, fp in enumerate(files):
        img = Image.open(fp).convert('RGB')
        x_list.append(np.array(img.resize((28, 28))))
        y_list.append(np.array(img.resize((56, 56))))
        if (i + 1) % 5000 == 0:
            print(f'  {i + 1}/{len(files)} images loaded...')
    x_train = np.array(x_list).astype('float32') / 255.
    y_train = np.array(y_list).astype('float32') / 255.
    x_train = x_train.transpose(0, 3, 1, 2)  # (N, H, W, 3) -> (N, 3, H, W)
    y_train = y_train.transpose(0, 3, 1, 2)  # (N, H, W, 3) -> (N, 3, H, W)
    np.save(cache_xtrain, x_train)
    np.save(cache_ytrain, y_train)
    print(f'Cached  x_train={x_train.shape}  y_train={y_train.shape}')

## 5. Load and Cache Test Data

In [ ]:
cache_xtest = os.path.join(PROCESSED, 'x_test_rgb.npy')
cache_ytest = os.path.join(PROCESSED, 'y_test_rgb.npy')

if os.path.exists(cache_xtest) and os.path.exists(cache_ytest):
    x_test = np.load(cache_xtest)
    y_test = np.load(cache_ytest)
    print(f'Loaded from cache  x_test={x_test.shape}  y_test={y_test.shape}')
else:
    x_list, y_list = [], []
    files = glob.glob(os.path.join(DATA_ROOT, 'test1', '*.jpg'))
    for i, fp in enumerate(files):
        img = Image.open(fp).convert('RGB')
        x_list.append(np.array(img.resize((28, 28))))
        y_list.append(np.array(img.resize((56, 56))))
        if (i + 1) % 2500 == 0:
            print(f'  {i + 1}/{len(files)} images loaded...')
    x_test = np.array(x_list).astype('float32') / 255.
    y_test = np.array(y_list).astype('float32') / 255.
    x_test = x_test.transpose(0, 3, 1, 2)   # (N, 3, 28, 28)
    y_test = y_test.transpose(0, 3, 1, 2)   # (N, 3, 56, 56)
    np.save(cache_xtest, x_test)
    np.save(cache_ytest, y_test)
    print(f'Cached  x_test={x_test.shape}  y_test={y_test.shape}')

## 6. Model Architecture

```
Input  (N, 3, 28, 28)          <- 3 channels (R, G, B)
  ↓ ENCODER
    Conv(3→32)+BN+ReLU, Conv(32→32)+BN+ReLU  →  MaxPool(2)  →  (N, 32, 14, 14)
    Conv(32→64)+BN+ReLU, Conv(64→64)+BN+ReLU  →  MaxPool(2)  →  (N, 64,  7,  7)
    Conv(64→128)+BN+ReLU, Dropout2d(0.25)                   →  (N,128,  7,  7)  [bottleneck]
  ↓ DECODER  (identical structure to grayscale)
    Conv(128→64)+BN+ReLU  →  Upsample(×2)                   →  (N, 64, 14, 14)
    Conv(64→32)+BN+ReLU   →  Upsample(×2)                   →  (N, 32, 28, 28)
    Conv(32→16)+BN+ReLU   →  Upsample(×2)                   →  (N, 16, 56, 56)
    Conv(16→3)+Sigmoid                                       →  (N,  3, 56, 56)  [output]
                                                                  ^
                                          3 output channels (R, G, B reconstructed)
```

**BatchNorm2d** is applied after every convolution (before ReLU) to stabilise training and accelerate convergence.

In [ ]:
class CnnUpscalerRGB(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 28x28 -> 14x14 -> 7x7
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Dropout2d(0.25),
        )
        # Decoder: 7x7 -> 14x14 -> 28x28 -> 56x56  (extra upsample = 2x SR)
        self.decoder = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(64, 32, 3, padding=1),  nn.BatchNorm2d(32), nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(32, 16, 3, padding=1),  nn.BatchNorm2d(16), nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(16, 3, 3, padding=1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = CnnUpscalerRGB().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

print(f'Running on: {device}')
print(model)

summary(model, input_size=(1, 3, 28, 28))

## 7. Training

- **Loss**: MSE averaged over all pixels and all 3 channels
- **Optimiser**: Adam (lr = 1e-3, default betas)
- **Scheduler**: ReduceLROnPlateau — halves LR if test MSE does not improve for 3 epochs
- **Batch size**: 128
- **Epochs**: configurable via `nepochs` (default 10)
- **Checkpoint**: best model (lowest train MSE) saved to `models/cnnUpscalingRGB.pth`

In [ ]:
dataset      = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
loader       = DataLoader(dataset, batch_size=128, shuffle=True)
test_dataset = TensorDataset(torch.tensor(x_test), torch.tensor(y_test))
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False)

os.makedirs('models', exist_ok=True)
best_loss    = float('inf')
train_losses = []
test_losses  = []

In [ ]:
nepochs = 10
for epoch in range(nepochs):
    # --- training pass ---
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(loader)

    # --- test pass (no gradient) ---
    model.eval()
    with torch.no_grad():
        epoch_test_loss = sum(
            criterion(model(X_b.to(device)), y_b.to(device)).item()
            for X_b, y_b in test_loader
        ) / len(test_loader)

    scheduler.step(epoch_test_loss)
    train_losses.append(epoch_loss)
    test_losses.append(epoch_test_loss)
    print(f'Epoch {epoch+1:02d}/{nepochs} | Train MSE: {epoch_loss:.6f} | Test MSE: {epoch_test_loss:.6f} | LR: {optimizer.param_groups[0]["lr"]:.6e}')
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), 'models/cnnUpscalingRGB.pth')
        print(f'           -> checkpoint saved (best={best_loss:.6f})')

## 8. Visualise Predictions

Because tensors are in CHW format `(N, 3, H, W)`, we call `.transpose(1, 2, 0)` to convert  
back to HWC `(H, W, 3)` for matplotlib display. `clip(0, 1)` guards against tiny numerical overshoot.

In [ ]:
model.load_state_dict(torch.load('models/cnnUpscalingRGB.pth', map_location=device))
model.eval()

n = 4
indices = np.random.choice(len(x_test), n, replace=False)

X_sample = torch.tensor(x_test[indices]).to(device)
with torch.no_grad():
    preds = model(X_sample).cpu().numpy()

fig, axes = plt.subplots(3, n, figsize=(8, 6))

for i in range(n):
    axes[0, i].imshow(x_test[indices[i]].transpose(1, 2, 0))
    axes[1, i].imshow(preds[i].transpose(1, 2, 0).clip(0, 1))
    axes[2, i].imshow(y_test[indices[i]].transpose(1, 2, 0))

    axes[0, i].set_title(f"Sample {i+1}", fontsize=10)

    axes[0, i].set_xlabel("Original resize (28X28)", fontsize=9)
    axes[1, i].set_xlabel("Upscale (56X56)", fontsize=9)
    axes[2, i].set_xlabel("Original Resize (56X56)", fontsize=9)

for ax in axes.flatten():
    ax.set_ylabel("")

plt.subplots_adjust(
    left=0.05, right=0.98, top=0.92, bottom=0.08, wspace=0.25, hspace=0.4
)
plt.show()

## 9. Train vs. Test Loss Curve

A **gap** between the two curves indicates overfitting (train loss much lower than test).  
Both curves tracking together indicates good generalisation.

In [ ]:
epochs_range = range(1, len(train_losses) + 1)
fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_range, train_losses, label='Train MSE', marker='o', markersize=3, linewidth=1.5)
ax.plot(epochs_range, test_losses,  label='Test MSE',  marker='s', markersize=3, linewidth=1.5,
        linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training vs. Test Loss - RGB Autoencoder')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()